# AI For Medicine
This notebook covers semantic image segmentation using a U-Net architecture and disease classification using convolutional neural networks paired with Grad-CAM visualizations.

In [ ]:
from itertools import chain
import opendatasets as od
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 1. Blood cell segmentation
**Dataset:** [Blood cell segmentation dataset](https://www.kaggle.com/datasets/jeetblahiri/bccd-dataset-with-mask).

See [UNet architecture](https://en.wikipedia.org/wiki/U-Net).

Key Metrics:

- [Precision](https://en.wikipedia.org/wiki/Precision_and_recall): Out of all pixels predicted as cells, how many were actually cells?
    $$\text{Precision} = \frac{TP}{TP + FP}$$
- [Recall](https://en.wikipedia.org/wiki/Precision_and_recall): Out of all actual cell pixels, how many did the model find?
    $$\text{Recall} = \frac{TP}{TP + FN}$$
- [Intersection over Union](https://medium.com/analytics-vidhya/iou-intersection-over-union-705a39e7acef) (IoU / Jaccard Index): Measures the overlap between the predicted mask ($A$) and ground truth mask ($B$).
    $$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

In [ ]:
# Download dataset
dataset_url = "https://www.kaggle.com/datasets/jeetblahiri/bccd-dataset-with-mask"
od.download(dataset_url, data_dir="./datasets")

In [ ]:
# Gather paths recursively to safely handle any subdirectory indexing variations
image_dir = (Path.cwd() / "datasets" / "bccd-dataset-with-mask" / "BCCD Dataset with mask" / "train" / "original")
image_paths = sorted(chain(image_dir.glob("*.png"),image_dir.glob("*.jpg")))
mask_paths = sorted((Path.cwd() / "datasets" / "bccd-dataset-with-mask" / "BCCD Dataset with mask" / "train" / "mask").glob("*.png"))

print(f"Total available Blood Cell Images: {len(image_paths)}")
print(f"Total available Annotation Masks: {len(mask_paths)}")
assert len(image_paths) == len(mask_paths), "Error: Images and Masks count do not match!"

# Lock in a standard 80% training / 20% validation distribution for model validation
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    image_paths, mask_paths, test_size=0.20, random_state=42
)

In [ ]:
# Custom PyTorch Dataset mapping
class BloodCellDataset(Dataset):
    def __init__(self, img_paths, mask_paths, target_size=(128, 128)):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.target_size = target_size

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load microscopic image (RGB - 3 Channels) and target mask (Grayscale - 1 Channel)
        img = Image.open(self.img_paths[idx]).convert('RGB').resize(self.target_size)
        mask = Image.open(self.mask_paths[idx]).convert('L').resize(self.target_size)

        # Convert to numpy and normalize pixels to [0, 1] range
        img_np = np.array(img, dtype=np.float32) / 255.0
        mask_np = np.array(mask, dtype=np.float32) / 255.0

        # Turn mask into true zero/one values (binary thresholding)
        mask_binary = (mask_np > 0.5).astype(np.float32)

        # Format axes to shape format required by PyTorch: (Channels, Height, Width)
        img_tensor = torch.tensor(img_np).permute(2, 0, 1)
        mask_tensor = torch.tensor(mask_binary).unsqueeze(0)

        return img_tensor, mask_tensor

# Generate active DataLoaders
train_dataset = BloodCellDataset(train_img_paths, train_mask_paths)
val_dataset = BloodCellDataset(val_img_paths, val_mask_paths)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
print("Data pipeline constructed successfully!")

In [ ]:
# U-Net Architecture Construction
class SimpleUNet(nn.Module):
    def __init__(self):
        super(SimpleUNet, self).__init__()

        # Encoder: Downsampling path (Input channels changed from 1 to 3 for RGB Blood Cells)
        self.enc1 = self.conv_block(3, 16)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self.conv_block(16, 32)
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck (lowest resolution layer)
        self.bottleneck = self.conv_block(32, 64)

        # Decoder: Upsampling path
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(64, 32) # Skip connection addition (32 + 32)
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(32, 16) # Skip connection addition (16 + 16)

        self.final_conv = nn.Conv2d(16, 1, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        u2 = torch.cat([u2, e2], dim=1) # Merging spatial data from encoder
        d2 = self.dec2(u2)

        u1 = self.up1(d2)
        u1 = torch.cat([u1, e1], dim=1) # Merging spatial data from encoder
        d1 = self.dec1(u1)

        return torch.sigmoid(self.final_conv(d1))

model = SimpleUNet()

In [ ]:
# CODE CELL: Model Training Execution Setup
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting demonstrative training for 3 epochs...")
model.train()
for epoch in range(3):
    running_loss = 0.0
    for imgs, msks in train_loader:
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, msks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/3 Completed - Average Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
# CODE CELL: Validation Sample Performance & Metric Computation
model.eval()
imgs, msks = next(iter(val_loader))

with torch.no_grad():
    preds = model(imgs)

# Pull first specimen index from validation set batch array
pred_binary = (preds[0, 0].numpy() > 0.5).astype(np.float32)
true_binary = msks[0, 0].numpy()
display_input = imgs[0].permute(1, 2, 0).numpy() # Convert back to (H, W, C) layout for presentation

# Math Calculations
tp = np.sum((pred_binary == 1) & (true_binary == 1))
fp = np.sum((pred_binary == 1) & (true_binary == 0))
fn = np.sum((pred_binary == 0) & (true_binary == 1))

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
iou = tp / (tp + fp + fn + 1e-8)

# Visualization Plot Setup
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(display_input)
axes[0].set_title("Input Stained Blood Image")
axes[0].axis('off')

axes[1].imshow(true_binary, cmap='gray')
axes[1].set_title("Ground Truth masks")
axes[1].axis('off')

axes[2].imshow(pred_binary, cmap='gray')
axes[2].set_title("U-Net AI Prediction")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"--- Refactored BCCD Evaluation Metrics On First Validation Sample ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"IoU Score: {iou:.4f}")

## 2. Pneumonia Classification with Grad-CAM
**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

In [ ]:
dataset_url = "https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia"
od.download(dataset_url, data_dir="./datasets")

In [ ]:
# Gather paths recursively to safely handle any subdirectory indexing variations
normal_paths = sorted((Path.cwd() / "datasets" / "chest-xray-pneumonia" / "chest_xray" / "train" / "NORMAL").glob("*.jpeg"))
normal_labels = [0 for i in range(len(normal_paths))]
pneumonia_paths = sorted((Path.cwd() / "datasets" / "chest-xray-pneumonia" / "chest_xray" / "train" / "PNEUMONIA").glob("*.jpeg"))
pneumonia_labels = [1 for i in range(len(pneumonia_paths))]
image_labels = normal_labels + pneumonia_labels
image_paths = normal_paths + pneumonia_paths
print(f"Total available Normal Images: {len(normal_paths)}")
print(f"Total available Pneumonia Images: {len(pneumonia_paths)}")

# Lock in a standard 80% training / 20% validation distribution for model validation
train_img_paths, val_img_paths, train_labels, val_labels = train_test_split(
    image_paths, image_labels, test_size=0.20, random_state=42
)
print(f"Train count: {len(train_img_paths)} / Validation count: {len(val_img_paths)}")

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, file_paths, labels, target_size=(128, 128)):
        self.file_paths = file_paths
        self.labels = labels
        self.target_size = target_size

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load real image, convert to Grayscale ('L') and resize
        img = Image.open(self.file_paths[idx]).convert('L').resize(self.target_size)

        # Normalize pixel intensities from [0, 255] to [0.0, 1.0]
        img_np = np.array(img, dtype=np.float32) / 255.0

        # Add channel dimension: (1, Height, Width) required by PyTorch Conv2d
        img_tensor = torch.tensor(img_np).unsqueeze(0)
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)

        return img_tensor, label_tensor

# Build active PyTorch loaders
train_dataset_xr = ChestXRayDataset(train_img_paths, train_labels)
val_dataset_xr = ChestXRayDataset(val_img_paths, val_labels)

train_loader_xr = DataLoader(train_dataset_xr, batch_size=16, shuffle=True)
val_loader_xr = DataLoader(val_dataset_xr, batch_size=16, shuffle=False)

In [ ]:
# CODE CELL
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super(PneumoniaCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # Target layer for Grad-CAM
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 2)
        )
        self.gradients = None

    def activations_hook(self, grad):
        self.gradients = grad

    def forward(self, x):
        x = self.features[0](x)
        x = self.features[1](x)
        x = self.features[2](x)

        # Target layer activation collection
        x = self.features[3](x)
        if x.requires_grad:
            h = x.register_hook(self.activations_hook)

        x = self.features[4](x)
        x = self.features[5](x)
        x = self.classifier(x)
        return x

    def get_activations_gradient(self):
        return self.gradients

    def get_activations(self, x):
        # Extract features up to target conv block
        return self.features[:4](x)

cls_model = PneumoniaCNN()

In [ ]:
# CODE CELL
criterion_xr = nn.CrossEntropyLoss()
optimizer_xr = optim.Adam(cls_model.parameters(), lr=0.0005)

print("Training model on real clinical images (Demonstration: 2 epochs)...")
cls_model.train()
for epoch in range(2):
    total_loss = 0.0
    for imgs, labels in train_loader_xr:
        optimizer_xr.zero_grad()
        outputs = cls_model(imgs)
        loss = criterion_xr(outputs, labels)
        loss.backward()
        optimizer_xr.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/2 Done | Loss: {total_loss/len(train_loader_xr):.4f}")

In [ ]:
# CODE CELL
def compute_gradcam(model, img_tensor, target_class):
    model.eval()
    # Explicitly clear gradients from previous iterations
    model.zero_grad()

    # Forward pass
    output = model(img_tensor.unsqueeze(0))

    # Backward pass for the specific target class
    output[:, target_class].backward()

    # Extract calculated gradients and feature maps
    gradients = model.get_activations_gradient()
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    activations = model.get_activations(img_tensor.unsqueeze(0)).detach()

    # Weight the feature maps by their gradient importances
    for i in range(activations.shape[1]):
        activations[:, i, :, :] *= pooled_gradients[i]

    heatmap = torch.mean(activations, dim=1).squeeze()
    heatmap = np.maximum(heatmap.numpy(), 0) # Apply ReLU
    heatmap /= np.max(heatmap) if np.max(heatmap) != 0 else 1
    return heatmap

# Locate real distinct validation cases to evaluate
normal_idx = val_labels.index(0)
pneumonia_idx = val_labels.index(1)

# Retrieve raw tensors directly from our true verification pipeline
normal_tensor, _ = val_dataset_xr[normal_idx]
pneumonia_tensor, _ = val_dataset_xr[pneumonia_idx]

# Setup target gradients
cls_model.train()
_ = cls_model(normal_tensor.unsqueeze(0))[:, 0].backward()

# Compute actual maps
heatmap_normal = compute_gradcam(cls_model, normal_tensor, target_class=0)
heatmap_pneumonia = compute_gradcam(cls_model, pneumonia_tensor, target_class=1)

# Plotting the real medical results side-by-side
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

# Displaying True Normal Patient Metrics
axes[0, 0].imshow(normal_tensor.squeeze(), cmap='gray')
axes[0, 0].set_title("Real Patient: Normal X-Ray")
axes[0, 0].axis('off')

axes[0, 1].imshow(normal_tensor.squeeze(), cmap='gray')
axes[0, 1].imshow(Image.fromarray(np.uint8(255*heatmap_normal)).resize((128,128)), cmap='jet', alpha=0.4)
axes[0, 1].set_title("Grad-CAM: Normal Class Attention")
axes[0, 1].axis('off')

# Displaying True Pneumonia Patient Metrics
axes[1, 0].imshow(pneumonia_tensor.squeeze(), cmap='gray')
axes[1, 0].set_title("Real Patient: Pneumonia X-Ray")
axes[1, 0].axis('off')

axes[1, 1].imshow(pneumonia_tensor.squeeze(), cmap='gray')
axes[1, 1].imshow(Image.fromarray(np.uint8(255*heatmap_pneumonia)).resize((128,128)), cmap='jet', alpha=0.4)
axes[1, 1].set_title("Grad-CAM: Explaining Pneumonia Focus")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()